In [0]:
CREATE OR REPLACE VIEW databricks_ws.gold.mart_company_performance AS
SELECT
    c.company_id,
    c.company_name,
    m.movie_id,
    m.title,
    t.release_year AS year_release_date,
    f.budget,
    f.revenue,

    CASE
        WHEN f.budget IS NULL OR f.budget <= 0 THEN 'Sin datos'
        WHEN f.budget < 10000000 THEN 'Bajo'
        WHEN f.budget < 50000000 THEN 'Medio'
        WHEN f.budget < 100000000 THEN 'Alto'
        ELSE 'Muy alto'
    END AS budget_range,

    CASE
        WHEN f.budget > 0 AND f.revenue > 0
        THEN ROUND(f.revenue / f.budget, 2)
    END AS revenue_multiple,

    CASE
        WHEN f.budget > 0 AND f.revenue > 0
        THEN CASE WHEN f.revenue >= f.budget * 2 THEN 1 ELSE 0 END
    END AS reached_2x

FROM databricks_ws.gold.fact_movies f

JOIN databricks_ws.gold.dim_movies m
    ON f.movie_id = m.movie_id

JOIN databricks_ws.gold.dim_time t
    ON f.time_id = t.time_id

JOIN databricks_ws.gold.bridge_movie_prod_companies mc
    ON f.movie_id = mc.movie_id

JOIN databricks_ws.gold.dim_prod_companies c
    ON mc.company_id = c.company_id;

In [0]:
SELECT
    company_name,
    COUNT(DISTINCT movie_id) AS total_movies,
    ROUND(MEDIAN(revenue_multiple), 2) AS median_multiple,
    ROUND(AVG(reached_2x) * 100, 2) AS success_rate_2x

FROM databricks_ws.gold.mart_company_performance

GROUP BY company_id, company_name
HAVING COUNT(reached_2x) >= 10
ORDER BY success_rate_2x DESC;

company_name,total_movies,median_multiple,success_rate_2x
Eon Productions,22,6.51,95.45
Pixar Animation Studios,16,4.18,93.75
Marvel Studios,13,4.18,92.31
Twentieth Century Fox Animation,11,4.86,90.91
Revolution Sun Studios,16,3.04,87.5
Blumhouse Productions,22,18.35,86.36
Platinum Dunes,13,5.2,84.62
Big Screen Productions,13,2.84,84.62
21 Laps Entertainment,12,2.75,83.33
"Kennedy/Marshall Company, The",12,2.46,83.33
